In [18]:
import os
import requests
import pandas as pd
import random
import time
import urllib3
from pathlib import Path
from evidently import Report
from evidently.presets import DataDriftPreset
from IPython.display import IFrame

In [4]:
URL = "https://$your_url/v2/models/$model/infer"
TOKEN = "$your_token"


In [7]:
os.makedirs("data/reports", exist_ok=True)

DATA_DIR = Path("./data")
REPORT_DIR = DATA_DIR / "reports"

REFERENCE_PATH = DATA_DIR / "reference_data.csv"
CURRENT_PATH = DATA_DIR / "current_data.csv"
REPORT_PATH = REPORT_DIR / "iris_drift_report.html"

In [48]:
reference = pd.read_csv(REFERENCE_PATH)
reference.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,4.4,2.9,1.4,0.2
1,4.9,2.5,4.5,1.7
2,6.8,2.8,4.8,1.4
3,4.9,3.1,1.5,0.1
4,5.5,2.5,4.0,1.3


In [49]:
# Suppress warnings because this demo connects to an internal
# OpenShift route using a self-signed/internal CA certificate.
urllib3.disable_warnings()

rows = []

for i in range(200):
    # intentionally drifted iris-like data
    sample = [
        random.uniform(6.5, 8.0),   # sepal length shifted higher
        random.uniform(2.0, 3.0),   # sepal width shifted lower
        random.uniform(4.5, 6.8),   # petal length shifted higher
        random.uniform(1.5, 2.5),
    ]
    payload = {
        "inputs": [{
            "name": "predict",
            "shape": [1, 4],
            "datatype": "FP32",
            "data": [sample]
        }]
    }
    r = requests.post(
        URL,
        headers={
            "Authorization": f"Bearer {TOKEN}",
            "Content-Type": "application/json"
        },
        json=payload,
        verify=False
    )

    result = r.json()
    
    prediction = result["outputs"][0]["data"][0]

    rows.append({
        "sepal_length": sample[0],
        "sepal_width": sample[1],
        "petal_length": sample[2],
        "petal_width": sample[3],
        "prediction": prediction
    })

    time.sleep(0.1)

current = pd.DataFrame(rows)
current = current.round({
    "sepal_length": 2,
    "sepal_width": 2,
    "petal_length": 2,
    "petal_width": 1
})
current.to_csv(CURRENT_PATH, index=False)

current.head()


,sepal_length,sepal_width,petal_length,petal_width,prediction
0,7.66,2.52,5.14,1.8,2
1,6.91,2.06,5.52,1.9,2
2,6.54,2.38,5.65,2.0,2
3,7.65,2.24,6.04,2.5,2
4,7.28,2.24,4.68,2.1,2


In [50]:
current.describe()

,sepal_length,sepal_width,petal_length,petal_width,prediction
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,7.283450,2.542000,5.621550,2.013000,1.980000
std,0.433879,0.269815,0.645134,0.299968,0.140351
min,6.510000,2.010000,4.510000,1.500000,1.000000
25%,6.920000,2.320000,5.110000,1.800000,2.000000
50%,7.360000,2.570000,5.555000,2.000000,2.000000
75%,7.660000,2.752500,6.170000,2.300000,2.000000
max,8.000000,2.990000,6.800000,2.500000,2.000000


In [51]:
print(reference.columns.tolist())
print(current.columns.tolist())

['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'prediction']


In [60]:
reference = reference.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
})

FEATURES = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
]

reference_features = reference[FEATURES]
current_features = current[FEATURES]

In [61]:
report = Report([
    DataDriftPreset()
])
my_eval = report.run(    
    reference_data=reference,
    current_data=current
)

my_eval.save_html(str(REPORT_PATH))
print("Exists:", REPORT_PATH.exists())
print("Path:", REPORT_PATH.resolve())

Exists: True
Path: /opt/app-root/src/data/reports/iris_drift_report.html


In [64]:
result_dict = my_eval.dict()
result_dict


{'metrics': [{'id': '15e89f895b482f9b84ba7274ed18a106',
   'metric_name': 'DriftedColumnsCount(drift_share=0.5)',
   'config': {'type': 'evidently:metric_v2:DriftedColumnsCount',
    'drift_share': 0.5},
   'value': {'count': 4.0, 'share': 1.0}},
  {'id': '2a802a425a158788ebfc9d82d0eddbdd',
   'metric_name': 'ValueDrift(column=sepal_length,method=K-S p_value,threshold=0.05)',
   'config': {'type': 'evidently:metric_v2:ValueDrift',
    'column': 'sepal_length',
    'method': 'K-S p_value',
    'threshold': 0.05},
   'value': np.float64(1.312057411945479e-48)},
  {'id': '8c08e50a5c960c67bc077126582e3c35',
   'metric_name': 'ValueDrift(column=sepal_width,method=K-S p_value,threshold=0.05)',
   'config': {'type': 'evidently:metric_v2:ValueDrift',
    'column': 'sepal_width',
    'method': 'K-S p_value',
    'threshold': 0.05},
   'value': np.float64(6.566684669420221e-25)},
  {'id': '15ab49c15df043ef6e5e213b0fe0b716',
   'metric_name': 'ValueDrift(column=petal_length,method=K-S p_value,thr

In [63]:
IFrame(str(REPORT_PATH), width=1200, height=800)